# jaxfne: Computational Biophysics Tutorial

**A 4-part journey: single neurons → cortical circuits → readouts → hypothesis tuning.**

> **Claim discipline (immutable throughout):** All outputs are *computational scaffolds*. Language rules enforced: `native current-like` (not physical current), `LFP-proxy` (not real LFP), `CSD-proxy` (not real CSD), `proxy` (not validated), `computational_scaffold` (not biological proof).

---

**Parts:**
1. Part 1 — Computational Models and Biophysics
2. Part 2 — Parallelization and Linear Algebra
3. Part 3 — Cortex: Building and Testing Readouts
4. Part 4 — Fine-Tuning for Hypothesis Tests
5. Export: Metrics, Tuning History, Manifest


## Learning Objectives

After completing this tutorial, you will understand:

1. **Source-generating neuron equations** — how Izhikevich-style reduced neuronal models emit voltage, spikes, and source-driving activity.

2. **Vectorized simulation** — how to scale from a single neuron to populations using JAX arrays, connectivity matrices, and batched circuit operations.

3. **Laminar cortical columns** — how to construct a cortical column with specified layers, cell types, connectivity structure, and 3D geometry.

4. **Multimodal proxy readouts** — how to extract MUA-proxy, LFP-proxy, CSD-proxy, EEG-proxy, MEG-proxy, EMM-proxy, and spectral features from simulated source dynamics.

5. **Hypothesis tuning** — how to formulate loss functions and optimize circuit parameters against rate, spectral, and laminar-profile targets.

In [ ]:
# Cell 0: Install and imports
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'jaxfne', 'numpy', 'scipy', 'pandas', 'matplotlib', 'scikit-learn'], check=True)

import os, json, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal

import jaxfne
import jax
import jax.numpy as jnp
from jaxfne import (
    make_eig_network, simulate_eig_izhikevich, IzhikevichParams,
    project_laminar_sources, CELL_TYPE_PRESETS,
)
from jaxfne.emitters import izhikevich_eig_params

print(f'jaxfne: {jaxfne.__version__}')
print(f'JAX:    {jax.__version__}')
print(f'Devices: {jax.devices()}')


In [ ]:
# Global config
SEED = 42
COLORS = {'E': '#2166ac', 'PV': '#d6604d', 'SST': '#4dac26', 'VIP': '#f4a742'}
LAYER_COLORS = {'L2/3': '#e41a1c', 'L4': '#377eb8', 'L5': '#4daf4a', 'L6': '#984ea3'}

DT_MS = 0.25
DURATION_MS = 1500.0   # -500 to +1000 ms
N_STEPS = int(DURATION_MS / DT_MS)
N_TOTAL = 100

T_AXIS = np.linspace(-500.0, 1000.0, N_STEPS)

FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name, fig=None):
    if fig is None:
        fig = plt.gcf()
    for ext in ['png', 'svg']:
        p = os.path.join(FIG_DIR, f'{name}.{ext}')
        kw = dict(bbox_inches='tight')
        if ext == 'png':
            kw['dpi'] = 150
        fig.savefig(p, **kw)
    plt.close(fig)
    print(f'  Saved: figures/{name}.png + .svg')

from matplotlib.lines import Line2D

print(f'Duration: {DURATION_MS} ms | dt: {DT_MS} ms | Steps: {N_STEPS}')
print(f'T_AXIS: {T_AXIS[0]:.0f} to {T_AXIS[-1]:.0f} ms')


---
## Part 1: Computational Models and Biophysics

> **Learning goal:** Understand what a model is, survey neural model families, and simulate a single neuron to produce voltage, spikes, and source proxy.


### 1.1 — What Is a Model?

A **computational model** is a set of mathematical rules that generate signals resembling those measured in neural recordings. Models are tools for *generating and testing hypotheses* — they are not proofs of biological mechanism.

Key ideas:
- A model neuron *emits* voltage-like state variables following defined dynamics.
- Its drive is a **native current-like** quantity — resembles ionic current in structure, but is **not calibrated to physical amperes**.
- Model outputs (spikes, source proxy, LFP-proxy signals) require explicit empirical validation before any biological claim can be made.

> **Claim gate:** All outputs in this tutorial are `computational_scaffold`. They are educational tools, not empirically validated biological signals.


### 1.2 — Neural Model Families

| Model | Dimensions | Key Feature | Typical Use |
|-------|-----------|-------------|-------------|
| **Hodgkin-Huxley (HH)** | ~4D/neuron | Explicit ion channel gating | Biophysical detail |
| **Izhikevich** | 2D (v, u) | Polynomial + reset, rich repertoire | Fast, diverse patterns |
| **QIF** | 1D + threshold | Canonical near bifurcation | Theoretical analysis |
| **LIF** | 1D + threshold | RC circuit approximation | Large-scale networks |
| **Rate model** | 1D/population | Mean firing rate | Population dynamics |

**jaxfne uses Izhikevich neurons** in this tutorial.

Dynamics: `dv/dt = 0.04v^2 + 5v + 140 - u + I_native`, `du/dt = a(bv - u)`, reset when `v >= 30 mV` (then `v <- c`, `u <- u + d`).

> Reference: Izhikevich, 2003. IEEE Trans. Neural Networks.


In [ ]:
# 1.3: Single-neuron Izhikevich simulation
key_p1 = jax.random.PRNGKey(SEED + 1)

params_single = izhikevich_eig_params(n=1, cell_type_fractions={'E': 1.0})
# Boost drive for clear spiking, no recurrence
params_single = IzhikevichParams(
    a=params_single.a, b=params_single.b, c=params_single.c, d=params_single.d,
    drive=jnp.array([8.0]),
    sign=params_single.sign,
    W=jnp.zeros((1, 1)),
    v0=params_single.v0, u0=params_single.u0,
    source_scale=params_single.source_scale,
    labels=params_single.labels,
    source_calibration_status='uncalibrated_izhikevich_native_current',
)

duration_single = 500.0
n_steps_single = int(duration_single / DT_MS)
v_single, spk_single, src_single = simulate_eig_izhikevich(
    params_single, n_steps=n_steps_single, dt_ms=DT_MS, key=key_p1
)

t_single = np.linspace(0, duration_single, n_steps_single)
v_s_np = np.array(v_single[:, 0])
spk_s_np = np.array(spk_single[:, 0])
src_s_np = np.array(src_single[:, 0])
spike_times_s = t_single[spk_s_np > 0.5]

print(f'Single E neuron | duration: {duration_single} ms | spikes: {len(spike_times_s)}')
print(f'Voltage range: [{v_s_np.min():.1f}, {v_s_np.max():.1f}] mV')
print(f'Source proxy range: [{src_s_np.min():.2f}, {src_s_np.max():.2f}]')
print('Source calibration: uncalibrated_izhikevich_native_current (proxy only)')


In [ ]:
# 1.4 + 1.5: Figures 01-03

# Figure 01: Voltage trace
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_single, v_s_np, color=COLORS['E'], lw=1.2)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Membrane potential (mV)')
ax.set_title('Figure 01 -- Single E Neuron Voltage Trace\n'
             '[Izhikevich | native current-like drive | computational scaffold]')
ax.text(0.01, 0.03, 'NOT physical mV amplitude | computational proxy',
        transform=ax.transAxes, fontsize=7, color='gray')
savefig('01_single_neuron_voltage', fig)

# Figure 02: Spike events
fig, ax = plt.subplots(figsize=(10, 1.5))
ax.eventplot(spike_times_s, lineoffsets=0, linelengths=0.8, colors=[COLORS['E']])
ax.set_xlabel('Time (ms)'); ax.set_yticks([])
ax.set_title('Figure 02 -- Single E Neuron Spike Events\n'
             '[spike detection at v >= 30 mV threshold | computational scaffold]')
savefig('02_single_neuron_spikes', fig)

# Figure 03: Source proxy
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_single, src_s_np, color='#8856a7', lw=1.0)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Native current-like proxy (a.u.)')
ax.set_title('Figure 03 -- Single Neuron Source Proxy\n'
             '[proxy_no_field_solve | uncalibrated_izhikevich_native_current]')
ax.text(0.01, 0.03, 'NOT physical current amplitude | computational proxy only',
        transform=ax.transAxes, fontsize=7, color='gray')
savefig('03_single_neuron_source_proxy', fig)

print('Figures 01-03 saved.')


---
## Part 2: Parallelization and Linear Algebra

> **Learning goal:** Scale from 1 neuron to N. Understand how connectivity matrices `W in R^{NxN}` encode circuit hypotheses. Simulate populations and visualize rasters and rates.


### 2.1 — From Scalar to Vector

**Single neuron:** `dv/dt = 0.04v^2 + 5v + 140 - u + I_drive + noise`

**N neurons (vectorized):**
```
dv/dt = 0.04v^2 + 5v + 140 - u + I_drive + W @ spikes_prev + noise
```

where `W in R^{NxN}` is the **connectivity matrix** encoding all pairwise synaptic weights.

**Why W matters:**
- `W[i, j] > 0`: neuron j excites neuron i (E pre-synaptic)
- `W[i, j] < 0`: neuron j inhibits neuron i (PV/SST/VIP pre-synaptic)
- `W[i, j] = 0`: no direct connection

> **Key insight:** Choosing W is choosing a circuit hypothesis. Simulation tests whether that hypothesis generates signals consistent with experimental data.


In [ ]:
# 2.2: Vectorized N-neuron population simulation
key_p2 = jax.random.PRNGKey(SEED + 2)
n_pop = 50
params_pop = izhikevich_eig_params(n=n_pop, cell_type_fractions={'E': 0.7, 'PV': 0.3})
duration_pop = 300.0
n_steps_pop = int(duration_pop / DT_MS)

v_pop, spk_pop, src_pop = simulate_eig_izhikevich(
    params_pop, n_steps=n_steps_pop, dt_ms=DT_MS, key=key_p2
)
v_pop_np = np.array(v_pop)
spk_pop_np = np.array(spk_pop)
t_pop = np.linspace(0, duration_pop, n_steps_pop)

labels_pop = list(params_pop.labels)
n_E_pop = labels_pop.count('E')
n_PV_pop = labels_pop.count('PV')
print(f'Population: {n_pop} neurons ({n_E_pop} E, {n_PV_pop} PV)')
print(f'Total spikes: {int(spk_pop_np.sum())}')
print(f'E firing rate: {spk_pop_np[:, :n_E_pop].sum() / (n_E_pop * duration_pop / 1000):.1f} Hz')
print(f'PV firing rate: {spk_pop_np[:, n_E_pop:].sum() / (n_PV_pop * duration_pop / 1000):.1f} Hz')


In [ ]:
# 2.3-2.5: Connectivity matrices — Figures 04-05
n_demo = 20
rng_conn = np.random.RandomState(SEED + 10)

# All-to-all mask
W_all = np.ones((n_demo, n_demo)) - np.eye(n_demo)

# Sparse connectivity p=0.1
W_sparse = (rng_conn.rand(n_demo, n_demo) < 0.1).astype(float)
np.fill_diagonal(W_sparse, 0)

# Figure 04: All-to-all
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(W_all, cmap='Blues', vmin=0, vmax=1)
ax.set_title('Figure 04 -- All-to-All Connectivity\n[W in R^{NxN} | circuit hypothesis]')
ax.set_xlabel('Pre-synaptic'); ax.set_ylabel('Post-synaptic')
plt.colorbar(im, ax=ax, fraction=0.046)
savefig('04_all_to_all', fig)

# Figure 05: Sparse
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(W_sparse, cmap='Greys', vmin=0, vmax=1)
ax.set_title(f'Figure 05 -- Sparse Connectivity (p=0.1)\nActual: {W_sparse.mean():.3f}')
ax.set_xlabel('Pre-synaptic'); ax.set_ylabel('Post-synaptic')
plt.colorbar(im, ax=ax, fraction=0.046)
savefig('05_sparse_mask', fig)

print('Connectivity matrices defined. Figures 04-05 saved.')


In [ ]:
# 2.6: Population raster + rate — Figures 06-07

# Figure 06: Raster
fig, ax = plt.subplots(figsize=(11, 5))
for nid in range(n_pop):
    ct = labels_pop[nid]
    spkt = t_pop[spk_pop_np[:, nid] > 0.5]
    ax.scatter(spkt, [nid]*len(spkt), c=COLORS.get(ct, '#999'), s=1.5, alpha=0.8)
leg = [Line2D([0],[0],marker='o',color='w',markerfacecolor=COLORS['E'],ms=6,label=f'E (n={n_E_pop})'),
       Line2D([0],[0],marker='o',color='w',markerfacecolor=COLORS['PV'],ms=6,label=f'PV (n={n_PV_pop})')]
ax.legend(handles=leg, loc='upper right', fontsize=8)
ax.axhline(n_E_pop-0.5, color='k', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Neuron ID')
ax.set_title('Figure 06 -- Population Raster (E/PV)\n'
             '[computational scaffold | native current-like drive]')
savefig('06_pop_raster', fig)

# Figure 07: Firing rate
bin_ms = 20.0
bedges = np.arange(0, duration_pop + bin_ms, bin_ms)
t_bins = 0.5*(bedges[:-1]+bedges[1:])
rate_E_p = []; rate_PV_p = []
for b in range(len(bedges)-1):
    m = (t_pop >= bedges[b]) & (t_pop < bedges[b+1])
    rate_E_p.append(spk_pop_np[m, :n_E_pop].sum() / (n_E_pop * bin_ms / 1000))
    rate_PV_p.append(spk_pop_np[m, n_E_pop:].sum() / (n_PV_pop * bin_ms / 1000))

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_bins, rate_E_p, color=COLORS['E'], lw=2, label='E (Hz)')
ax.plot(t_bins, rate_PV_p, color=COLORS['PV'], lw=2, ls='--', label='PV (Hz)')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Firing rate (Hz)')
ax.set_title('Figure 07 -- Population Firing Rate (20 ms bins)\n'
             '[computational scaffold | proxy firing rates]')
ax.legend(fontsize=9)
savefig('07_pop_rate', fig)

print('Figures 06-07 saved.')


---
## Part 3: Cortex — Building and Testing Readouts

> **Learning goal:** Build a laminar cortical column with E, PV, SST, VIP across 4 layers. Simulate spontaneous and evoked activity. Apply readout operators: LFP-proxy, CSD-proxy, TFR, bandpower.


### 3.1 — Laminar Cortical Column

**Cell types and roles:**
| Cell type | Symbol | n | Role |
|-----------|--------|---|------|
| Excitatory | E (large triangle) | 50 | Regular-spiking principal cells |
| Parvalbumin+ | PV (small circle) | 25 | Fast perisomatic inhibition |
| Somatostatin+ | SST (small circle) | 15 | Dendritic inhibition |
| VIP+ | VIP (small circle) | 10 | Disinhibitory interneurons |

**Layer assignments:** L2/3 (superficial) → L4 → L5 → L6 (deep)

> Claim: This is a **computational scaffold** for learning circuit concepts. Population sizes are educational, not empirically calibrated.


In [ ]:
# 3.1: Build laminar cortical column
N_E = 50; N_PV = 25; N_SST = 15; N_VIP = 10
assert N_E + N_PV + N_SST + N_VIP == N_TOTAL

LAYER_BOUNDS = {
    'L2/3': (0.00, 0.25),
    'L4':   (0.25, 0.50),
    'L5':   (0.50, 0.75),
    'L6':   (0.75, 1.00),
}

cell_type_fracs = {'E': N_E/N_TOTAL, 'PV': N_PV/N_TOTAL,
                   'SST': N_SST/N_TOTAL, 'VIP': N_VIP/N_TOTAL}
params_ctx = izhikevich_eig_params(n=N_TOTAL, cell_type_fractions=cell_type_fracs)
labels_ctx = list(params_ctx.labels)

positions_ctx = np.zeros((N_TOTAL, 3))
rng_pos = np.random.RandomState(SEED + 20)

type_indices = {}
cum = 0
for ct, n_ct in [('E', N_E), ('PV', N_PV), ('SST', N_SST), ('VIP', N_VIP)]:
    idx = list(range(cum, cum + n_ct))
    type_indices[ct] = idx
    if ct == 'E':      depths = np.sort(rng_pos.uniform(0.2, 1.0, n_ct))
    elif ct == 'PV':   depths = np.sort(rng_pos.uniform(0.0, 1.0, n_ct))
    elif ct == 'SST':  depths = np.sort(rng_pos.uniform(0.0, 0.6, n_ct))
    else:              depths = np.sort(rng_pos.uniform(0.0, 0.35, n_ct))
    positions_ctx[idx, 0] = rng_pos.uniform(-0.1, 0.1, n_ct)
    positions_ctx[idx, 2] = depths
    cum += n_ct

print('Cortical column:')
for ct, idx in type_indices.items():
    d = positions_ctx[idx, 2]
    print(f'  {ct}: n={len(idx)}, depth=[{d.min():.2f}, {d.max():.2f}]')


In [ ]:
# Figure 08: Column layout
fig, ax = plt.subplots(figsize=(5, 8))
for ct, idx in type_indices.items():
    x = positions_ctx[idx, 0]
    y = 1.0 - positions_ctx[idx, 2]
    sz = 80 if ct == 'E' else 35
    mk = '^' if ct == 'E' else 'o'
    ax.scatter(x, y, c=COLORS[ct], s=sz, marker=mk, alpha=0.7,
               label=f'{ct} (n={len(idx)})', edgecolors='k', linewidths=0.3)
for layer, (lo, hi) in LAYER_BOUNDS.items():
    y_mid = 1.0 - 0.5*(lo+hi)
    ax.axhline(1.0-lo, color='gray', lw=0.8, ls='--', alpha=0.6)
    ax.text(0.15, y_mid, layer, ha='left', va='center', fontsize=11,
            color=LAYER_COLORS[layer], fontweight='bold')
ax.set_xlim(-0.2, 0.2)
ax.set_xlabel('Lateral position (a.u.)')
ax.set_ylabel('Cortical depth (top=superficial)')
ax.set_title('Figure 08 -- Laminar Cortical Column Layout\n'
             '[E: triangles | IN: circles | computational scaffold]')
ax.legend(loc='lower right', fontsize=8)
savefig('08_layout', fig)
print('Figure 08 saved.')


In [ ]:
# 3.4: Spontaneous activity simulation
key_spont = jax.random.PRNGKey(SEED + 3)
print(f'Simulating {DURATION_MS} ms spontaneous ({N_STEPS} steps)...')
t0 = time.time()

v_ctx, spk_ctx, src_ctx = simulate_eig_izhikevich(
    params_ctx, n_steps=N_STEPS, dt_ms=DT_MS, key=key_spont
)

print(f'Done in {time.time()-t0:.1f} s')
v_ctx_np = np.array(v_ctx)
spk_ctx_np = np.array(spk_ctx)
src_ctx_np = np.array(src_ctx)

def fr_hz(spk, idx, dur_ms):
    if not idx: return 0.0
    return float(spk[:, idx].sum()) / (len(idx) * dur_ms / 1000)

print('Spontaneous firing rates:')
for ct in ['E','PV','SST','VIP']:
    print(f'  {ct}: {fr_hz(spk_ctx_np, type_indices[ct], DURATION_MS):.2f} Hz')


In [ ]:
# Figure 09: Spontaneous raster
fig, ax = plt.subplots(figsize=(12, 5))
for ct in ['E','PV','SST','VIP']:
    for nid in type_indices[ct]:
        sm = spk_ctx_np[:, nid] > 0.5
        ax.scatter(T_AXIS[sm], [nid]*sm.sum(), c=COLORS[ct], s=1.0, alpha=0.7)
ax.axvline(0, color='r', lw=1.0, ls='--', label='t=0 (event onset)')
for b in [N_E, N_E+N_PV, N_E+N_PV+N_SST]:
    ax.axhline(b-0.5, color='k', lw=0.5, ls='--', alpha=0.4)
leg = [Line2D([0],[0],marker='o',color='w',markerfacecolor=COLORS[ct],ms=6,label=ct)
       for ct in ['E','PV','SST','VIP']]
ax.legend(handles=leg+[Line2D([0],[0],color='r',lw=1.5,ls='--',label='t=0')],
          loc='upper right', fontsize=7)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Neuron ID')
ax.set_title('Figure 09 -- Spontaneous Raster (no evoked drive)\n'
             '[computational scaffold | native current-like drive]')
ax.set_xlim(T_AXIS[0], T_AXIS[-1])
savefig('09_spont_raster', fig)

# Figure 10: Spontaneous pop rate
bin_ms_c = 25.0
bedges_c = np.arange(T_AXIS[0], T_AXIS[-1]+bin_ms_c, bin_ms_c)
t_bins_c = 0.5*(bedges_c[:-1]+bedges_c[1:])
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for i, ct in enumerate(['E','PV','SST','VIP']):
    idx = type_indices[ct]
    rates = [spk_ctx_np[(T_AXIS>=bedges_c[b])&(T_AXIS<bedges_c[b+1])][:,idx].sum()
             / (len(idx)*bin_ms_c/1000) if idx else 0
             for b in range(len(bedges_c)-1)]
    axes[i].plot(t_bins_c, rates, color=COLORS[ct], lw=1.5)
    axes[i].axvline(0, color='r', lw=0.8, ls='--', alpha=0.6)
    axes[i].set_ylabel(f'{ct}\n(Hz)', fontsize=8)
    axes[i].set_ylim(bottom=0)
axes[-1].set_xlabel('Time (ms)')
axes[0].set_title('Figure 10 -- Spontaneous Population Firing Rates (25 ms bins)')
plt.tight_layout()
savefig('10_spont_rate', fig)
print('Figures 09-10 saved.')


In [ ]:
# 3.5: Laminar field projections
positions_jax = jnp.array(positions_ctx)
n_contacts = 16

field_out = project_laminar_sources(
    src_ctx_np, positions_jax, n_contacts=n_contacts, width=0.08
)
lfp_like = np.array(field_out.lfp_proxy)   # [T, n_contacts]
csd_like = np.array(field_out.csd_proxy)   # [T, n_contacts]

print(f'LFP-proxy shape:  {lfp_like.shape}')
print(f'CSD-proxy shape:  {csd_like.shape}')
print('Source calibration: uncalibrated_izhikevich_native_current')
print('Readout type: LFP-proxy (NOT real LFP) | CSD-proxy (NOT real CSD)')

# Figure 11: LFP-proxy laminar probe
fig, ax = plt.subplots(figsize=(12, 6))
spacing = np.abs(lfp_like).max() * 1.5 + 0.01
for c in range(n_contacts):
    ax.plot(T_AXIS, lfp_like[:, c] + (n_contacts-c)*spacing, color='steelblue', lw=0.6, alpha=0.8)
ax.axvline(0, color='r', lw=1.0, ls='--', label='t=0')
ax.set_xlabel('Time (ms)'); ax.set_yticks([]); ax.set_ylabel('Channels (top=superficial)')
ax.set_title('Figure 11 -- LFP-proxy Laminar Probe (16 contacts)\n'
             '[proxy_no_field_solve | NOT real LFP | computational scaffold]')
ax.text(0.01, 0.01, 'LFP-proxy: native current-like source | not validated',
        transform=ax.transAxes, fontsize=7, color='gray')
ax.legend(fontsize=8)
savefig('11_lfp_proxy', fig)
print('Figure 11 saved.')


In [ ]:
# Figure 12: CSD-proxy
t_mask_plot = (T_AXIS >= -200) & (T_AXIS <= 700)
t_plot_short = T_AXIS[t_mask_plot]
csd_plot = csd_like[t_mask_plot, :]
vmax_c = np.percentile(np.abs(csd_plot), 98)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(csd_plot.T, aspect='auto', cmap='RdBu_r',
               vmin=-vmax_c, vmax=vmax_c,
               extent=[t_plot_short[0], t_plot_short[-1], n_contacts, 0])
plt.colorbar(im, ax=ax, label='CSD-proxy (a.u.)')
ax.axvline(0, color='k', lw=1.0, ls='--', label='t=0')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Contact (top=superficial)')
ax.set_title('Figure 12 -- CSD-proxy (spatial derivative of LFP-proxy)\n'
             '[NOT real CSD | proxy_no_field_solve | computational scaffold]')
ax.legend(fontsize=8)
savefig('12_csd_proxy', fig)
print('Figure 12 saved.')


In [ ]:
# 3.6: TFR and bandpower — Figures 13-14
lfp_mean = lfp_like.mean(axis=1)
fs_hz = 1000.0 / DT_MS
nperseg = min(256, N_STEPS // 4)
f_tfr, t_tfr_raw, Sxx = scipy_signal.spectrogram(
    lfp_mean, fs=fs_hz, nperseg=nperseg,
    noverlap=nperseg - int(nperseg*0.9), scaling='density'
)
t_tfr_ms = t_tfr_raw * 1000.0 + T_AXIS[0]
f_mask = f_tfr <= 80
Sxx_plot = Sxx[f_mask, :]
f_plot = f_tfr[f_mask]

def bandpower_db(psd, freqs, fmin, fmax):
    m = (freqs >= fmin) & (freqs <= fmax)
    if m.sum() == 0: return np.zeros(psd.shape[1])
    return 10.0 * np.log10(np.maximum(psd[m, :].mean(axis=0), 1e-30))

ab_db_ts = bandpower_db(Sxx[f_mask], f_plot, 8, 30)
gm_db_ts = bandpower_db(Sxx[f_mask], f_plot, 30, 80)

# Figure 13: TFR
fig, ax = plt.subplots(figsize=(12, 5))
S_db = 10*np.log10(np.maximum(Sxx_plot, 1e-30))
vmax_t = np.percentile(S_db, 99); vmin_t = vmax_t - 30
im = ax.pcolormesh(t_tfr_ms, f_plot, S_db, cmap='viridis',
                   vmin=vmin_t, vmax=vmax_t, shading='gouraud')
plt.colorbar(im, ax=ax, label='Power (dB)')
ax.axvline(0, color='w', lw=1.0, ls='--', label='t=0')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Frequency (Hz)')
ax.set_title('Figure 13 -- TFR (STFT of LFP-proxy mean)\n'
             '[proxy readout | NOT validated spectral content | computational scaffold]')
ax.legend(fontsize=8)
savefig('13_tfr', fig)

# Figure 14: Bandpower timeseries
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t_tfr_ms, ab_db_ts, color='#d62728', lw=1.5, label='Alpha/beta (8-30 Hz)')
axes[0].axvline(0, color='k', lw=0.8, ls='--')
axes[0].set_ylabel('Power (dB)'); axes[0].legend(fontsize=8)
axes[1].plot(t_tfr_ms, gm_db_ts, color='#9467bd', lw=1.5, label='Gamma (30-80 Hz)')
axes[1].axvline(0, color='k', lw=0.8, ls='--')
axes[1].set_ylabel('Power (dB)'); axes[1].legend(fontsize=8)
axes[1].set_xlabel('Time (ms)')
axes[0].set_title('Figure 14 -- Alpha/Beta and Gamma Bandpower\n'
                  '[LFP-proxy | NOT empirically calibrated | computational scaffold]')
plt.tight_layout()
savefig('14_bandpower', fig)
print('Figures 13-14 saved.')


In [ ]:
# 3.7: Evoked L4 drive — Figures 15-16
L4_idx = [i for i in range(N_TOTAL) if 0.25 <= positions_ctx[i, 2] < 0.50]
print(f'L4 neurons: {len(L4_idx)}')

evoked_amp = 6.0
drive_sched = np.zeros((N_STEPS, N_TOTAL), dtype=np.float32)
t0_i = int(500.0 / DT_MS)   # t=0 relative = 500ms absolute
t500_i = int(1000.0 / DT_MS) # t=500ms relative
drive_sched[t0_i:t500_i, L4_idx] = evoked_amp

key_evk = jax.random.PRNGKey(SEED + 4)
print('Simulating evoked response...')
t_e0 = time.time()
v_evk, spk_evk, src_evk = simulate_eig_izhikevich(
    params_ctx, n_steps=N_STEPS, dt_ms=DT_MS, key=key_evk,
    drive_schedule=jnp.array(drive_sched)
)
print(f'Done in {time.time()-t_e0:.1f} s')
v_evk_np = np.array(v_evk)
spk_evk_np = np.array(spk_evk)
src_evk_np = np.array(src_evk)

t_mask_ev = (T_AXIS >= 0) & (T_AXIS < 500)
print('Evoked (0-500 ms) firing rates:')
for ct in ['E','PV','SST','VIP']:
    idx = type_indices[ct]
    r = spk_evk_np[t_mask_ev][:, idx].sum() / (len(idx) * 500/1000) if idx else 0
    print(f'  {ct}: {r:.2f} Hz')


In [ ]:
# Figure 15: Evoked raster
fig, ax = plt.subplots(figsize=(12, 5))
for ct in ['E','PV','SST','VIP']:
    for nid in type_indices[ct]:
        sm = spk_evk_np[:, nid] > 0.5
        ax.scatter(T_AXIS[sm], [nid]*sm.sum(), c=COLORS[ct], s=1.0, alpha=0.7)
ax.axvline(0, color='r', lw=1.2, ls='--', label='L4 drive onset')
ax.axvline(500, color='orange', lw=1.0, ls='--', alpha=0.8, label='L4 drive offset')
for b in [N_E, N_E+N_PV, N_E+N_PV+N_SST]:
    ax.axhline(b-0.5, color='k', lw=0.5, ls='--', alpha=0.4)
leg = [Line2D([0],[0],marker='o',color='w',markerfacecolor=COLORS[ct],ms=6,label=ct)
       for ct in ['E','PV','SST','VIP']]
leg += [Line2D([0],[0],color='r',lw=1.5,ls='--',label='Onset'),
        Line2D([0],[0],color='orange',lw=1.2,ls='--',label='Offset')]
ax.legend(handles=leg, loc='upper left', fontsize=7, ncol=2)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Neuron ID')
ax.set_title('Figure 15 -- Evoked Raster (L4 native current-like drive 0-500 ms)\n'
             '[computational scaffold]')
ax.set_xlim(T_AXIS[0], T_AXIS[-1])
savefig('15_evoked_raster', fig)

# Figure 16: Evoked LFP-proxy
field_evk = project_laminar_sources(src_evk_np, positions_jax, n_contacts=n_contacts, width=0.08)
lfp_evk = np.array(field_evk.lfp_proxy)
fig, ax = plt.subplots(figsize=(12, 6))
sp_e = np.abs(lfp_evk).max()*1.5+0.01
for c in range(n_contacts):
    ax.plot(T_AXIS, lfp_evk[:, c]+(n_contacts-c)*sp_e, color='steelblue', lw=0.6, alpha=0.8)
ax.axvline(0, color='r', lw=1.0, ls='--', label='L4 onset')
ax.axvline(500, color='orange', lw=0.8, ls='--', alpha=0.7, label='L4 offset')
ax.set_xlabel('Time (ms)'); ax.set_yticks([])
ax.set_title('Figure 16 -- Evoked LFP-proxy Laminar Probe\n'
             '[proxy_no_field_solve | NOT real LFP | computational scaffold]')
ax.legend(fontsize=8)
savefig('16_evoked_lfp', fig)
print('Figures 15-16 saved.')


In [ ]:
# Figure 17: Evoked vs Spontaneous bandpower
lfp_evk_mean = lfp_evk.mean(axis=1)
lfp_spont_mean = lfp_like.mean(axis=1)

_, _, Sxx_evk2 = scipy_signal.spectrogram(
    lfp_evk_mean, fs=fs_hz, nperseg=nperseg,
    noverlap=nperseg-int(nperseg*0.9), scaling='density')
f_s2, _, Sxx_spont2 = scipy_signal.spectrogram(
    lfp_spont_mean, fs=fs_hz, nperseg=nperseg,
    noverlap=nperseg-int(nperseg*0.9), scaling='density')
f2_mask = f_s2 <= 80

psd_evk_db = 10*np.log10(np.maximum(Sxx_evk2[f2_mask,:].mean(axis=1), 1e-30))
psd_spont_db = 10*np.log10(np.maximum(Sxx_spont2[f2_mask,:].mean(axis=1), 1e-30))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(f_s2[f2_mask], psd_spont_db, color='steelblue', lw=2, label='Spontaneous')
ax.plot(f_s2[f2_mask], psd_evk_db, color='red', lw=2, ls='--', label='Evoked (L4)')
ax.axvspan(8, 30, alpha=0.1, color='blue', label='Alpha/beta')
ax.axvspan(30, 80, alpha=0.1, color='purple', label='Gamma')
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Power (dB)')
ax.set_title('Figure 17 -- Evoked vs Spontaneous Bandpower\n'
             '[LFP-proxy PSD | NOT calibrated | computational scaffold]')
ax.legend(fontsize=8)
savefig('17_evoked_vs_spont_bandpower', fig)
print('Figure 17 saved. Part 3 complete.')


---
## Part 4: Fine-Tuning for Hypothesis Tests

> **Learning goal:** Formulate a tuning objective, run a GSDR-style random search (15 steps), track loss and parameter trajectories, compare pre/post rasters, TFR, bandpower.

> **Claim gate:** This is a **demo-scale optimization** (15 steps). It does not guarantee convergence, does not produce biologically validated parameters, and does not claim empirical calibration.


### 4.1 — Hypothesis and Objective

**Hypothesis:** Tuning `{l4_drive, e_gain, pv_gain, sst_gain, vip_gain}` can bring firing rates and spectral balance closer to educational targets.

**Loss function:**
```
loss = w_rate * rate_error + w_bandpower * bandpower_error + w_sync * sync_penalty
```

**Parameters:**
- `l4_drive` in [2, 12]: L4 evoked native current-like amplitude
- `e_gain` in [0.5, 2]: E drive multiplier
- `pv_gain` in [0.5, 2]: PV drive multiplier
- `sst_gain` in [0.5, 2]: SST drive multiplier
- `vip_gain` in [0.5, 2]: VIP drive multiplier

> Gains multiply the Izhikevich native current-like drive — not physical units.


In [ ]:
# 4.1: Define metrics and loss
TARGET_RATES = {'E': 10.0, 'PV': 15.0, 'SST': 8.0, 'VIP': 6.0}
OBJ_WEIGHTS = {'rate': 1.0, 'bandpower': 0.5, 'sync_penalty': 2.0}
PARAM_BOUNDS = {
    'l4_drive': (2.0, 12.0),
    'e_gain':   (0.5, 2.0),
    'pv_gain':  (0.5, 2.0),
    'sst_gain': (0.5, 2.0),
    'vip_gain': (0.5, 2.0),
}

def compute_metrics(spk_arr, src_arr, t_ax, type_idx, pos_jax, nc=16, dt=DT_MS):
    fsh = 1000.0 / dt
    t_ev = (t_ax >= 0) & (t_ax < 500)
    rates = {}
    for ct in ['E','PV','SST','VIP']:
        idx = type_idx[ct]
        rates[ct] = float(spk_arr[t_ev][:,idx].sum()) / (len(idx)*500/1000) if idx else 0.0
    field = project_laminar_sources(src_arr, pos_jax, n_contacts=nc, width=0.08)
    lfp_m = np.array(field.lfp_proxy).mean(axis=1)
    nps = min(256, len(lfp_m)//4)
    f_s, _, Sx = scipy_signal.spectrogram(lfp_m, fs=fsh, nperseg=nps,
                                            noverlap=nps-int(nps*0.9), scaling='density')
    ab = float(np.mean(Sx[(f_s>=8)&(f_s<=30), :]))
    gm = float(np.mean(Sx[(f_s>=30)&(f_s<=80), :]))
    ab_db = 10*np.log10(max(ab, 1e-30))
    gm_db = 10*np.log10(max(gm, 1e-30))
    eidx = type_idx['E']
    if len(eidx) > 1:
        bsz = int(50.0/dt)
        nb = len(t_ax)//bsz
        sb = spk_arr[:nb*bsz, :][:,eidx].reshape(nb, bsz, -1).sum(axis=1)
        sync = 0.0 if sb.std(axis=0).min() < 1e-8 else float(np.abs(np.corrcoef(sb.T)[~np.eye(len(eidx),dtype=bool)]).mean())
    else:
        sync = 0.0
    return {'rates': rates, 'ab_db': ab_db, 'gm_db': gm_db, 'sync': sync}

def compute_loss(mets):
    re = sum(((mets['rates'][ct]-t)/max(t,1))**2 for ct,t in TARGET_RATES.items()) / len(TARGET_RATES)
    be = max(0.0, mets['ab_db'] - mets['gm_db'] + 2.0) / 10.0
    sp = max(0.0, mets['sync'] - 0.3)
    return float(OBJ_WEIGHTS['rate']*re + OBJ_WEIGHTS['bandpower']*be + OBJ_WEIGHTS['sync_penalty']*sp)

print('Objective defined. Targets:', TARGET_RATES)
print('Weights:', OBJ_WEIGHTS)


In [ ]:
# 4.2: Apply params and simulate helper
def run_with_params(params_base, pdict, n_steps, dt, key, l4_idx):
    g = np.ones(N_TOTAL, np.float32)
    for i, ct in enumerate(params_base.labels):
        g[i] = pdict.get(ct.lower()+'_gain', pdict.get('e_gain', 1.0)) if ct == 'E' \
            else pdict.get(ct.lower()+'_gain', 1.0)
    # Build gain map
    gain_map = {'E': pdict['e_gain'], 'PV': pdict['pv_gain'],
                'SST': pdict['sst_gain'], 'VIP': pdict['vip_gain']}
    for i, ct in enumerate(params_base.labels):
        g[i] = gain_map.get(ct, 1.0)
    new_drive = params_base.drive * jnp.array(g)
    mp = IzhikevichParams(
        a=params_base.a, b=params_base.b, c=params_base.c, d=params_base.d,
        drive=new_drive, sign=params_base.sign, W=params_base.W,
        v0=params_base.v0, u0=params_base.u0, source_scale=params_base.source_scale,
        labels=params_base.labels,
        source_calibration_status='uncalibrated_izhikevich_native_current',
    )
    sched = np.zeros((n_steps, N_TOTAL), np.float32)
    t0i = int(500/dt); t500i = int(1000/dt)
    sched[t0i:t500i, l4_idx] = pdict['l4_drive']
    _, spk, src = simulate_eig_izhikevich(mp, n_steps=n_steps, dt_ms=dt, key=key,
                                           drive_schedule=jnp.array(sched))
    return np.array(spk), np.array(src)

# Evaluate initial
init_p = {'l4_drive':6.0,'e_gain':1.0,'pv_gain':1.0,'sst_gain':1.0,'vip_gain':1.0}
key_init = jax.random.PRNGKey(SEED+50)
spk0, src0 = run_with_params(params_ctx, init_p, N_STEPS, DT_MS, key_init, L4_idx)
m0 = compute_metrics(spk0, src0, T_AXIS, type_indices, positions_jax)
l0 = compute_loss(m0)
print(f'Initial loss: {l0:.4f}')
print(f'Initial rates: {m0["rates"]}')
print(f'Initial AB/Gamma: {m0["ab_db"]:.2f} / {m0["gm_db"]:.2f} dB')
print(f'Initial sync: {m0["sync"]:.3f}')


In [ ]:
# 4.3-4.4: GSDR-style random search (15 steps)
N_OPT = 15
rng_opt = np.random.RandomState(SEED+99)

history = [{'step':0,'loss':l0,**init_p,
             'e_rate':m0['rates']['E'],'pv_rate':m0['rates']['PV'],
             'sst_rate':m0['rates']['SST'],'vip_rate':m0['rates']['VIP'],
             'ab_db':m0['ab_db'],'gm_db':m0['gm_db'],'sync':m0['sync']}]

best_loss = l0; best_p = dict(init_p); best_m = m0
best_spk = spk0; best_src = src0
bounds = dict(PARAM_BOUNDS)

print(f'Running {N_OPT} optimization steps...')
for step in range(1, N_OPT+1):
    cands = [{k: rng_opt.uniform(lo,hi) for k,(lo,hi) in bounds.items()} for _ in range(4)]
    ks = jax.random.PRNGKey(SEED+100+step)
    step_best = best_loss; step_p = best_p; step_m = best_m
    step_spk = best_spk; step_src = best_src
    for c in cands:
        sp_c, sr_c = run_with_params(params_ctx, c, N_STEPS, DT_MS, ks, L4_idx)
        mc = compute_metrics(sp_c, sr_c, T_AXIS, type_indices, positions_jax)
        lc = compute_loss(mc)
        if lc < step_best:
            step_best=lc; step_p=c; step_m=mc; step_spk=sp_c; step_src=sr_c
    best_loss=step_best; best_p=step_p; best_m=step_m
    best_spk=step_spk; best_src=step_src
    # Narrow bounds
    new_bounds = {}
    for k,(lo,hi) in bounds.items():
        bv=best_p[k]; w=(hi-lo)*0.8
        new_bounds[k] = (max(lo, bv-w/2), min(hi, bv+w/2))
    bounds = new_bounds
    history.append({'step':step,'loss':best_loss,**best_p,
                    'e_rate':best_m['rates']['E'],'pv_rate':best_m['rates']['PV'],
                    'sst_rate':best_m['rates']['SST'],'vip_rate':best_m['rates']['VIP'],
                    'ab_db':best_m['ab_db'],'gm_db':best_m['gm_db'],'sync':best_m['sync']})
    if step%5==0 or step==N_OPT:
        print(f'  Step {step:2d}: loss={best_loss:.4f} E={best_m["rates"]["E"]:.1f} PV={best_m["rates"]["PV"]:.1f} sync={best_m["sync"]:.3f}')

tuning_df = pd.DataFrame(history)
print(f'\nBest loss: {best_loss:.4f}')
print(f'Best params: {best_p}')
print(f'Best rates: {best_m["rates"]}')


In [ ]:
# Figures 18-19: Loss curve and parameter trajectory

# Figure 18: Loss curve
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tuning_df['step'], tuning_df['loss'], 'o-', color='#e31a1c', lw=2, ms=5)
ax.set_xlabel('Optimization step'); ax.set_ylabel('Loss')
ax.set_title('Figure 18 -- Tuning Loss Curve (GSDR-style random search)\n'
             '[15-step demo | NOT converged | computational scaffold]')
savefig('18_loss_curve', fig)

# Figure 19: Parameter trajectory
pnames = ['l4_drive','e_gain','pv_gain','sst_gain','vip_gain']
pcols  = [COLORS['PV'],COLORS['E'],COLORS['PV'],COLORS['SST'],COLORS['VIP']]
fig, axes = plt.subplots(5,1,figsize=(10,10),sharex=True)
for i,(pn,pc) in enumerate(zip(pnames,pcols)):
    axes[i].plot(tuning_df['step'], tuning_df[pn], 'o-', color=pc, lw=1.8, ms=4)
    axes[i].set_ylabel(pn, fontsize=8)
    lo,hi = PARAM_BOUNDS[pn]
    axes[i].set_ylim(lo*0.9, hi*1.05)
axes[-1].set_xlabel('Optimization step')
axes[0].set_title('Figure 19 -- Parameter Trajectory\n'
                  '[native current-like gains | computational scaffold]')
plt.tight_layout()
savefig('19_param_trajectory', fig)
print('Figures 18-19 saved.')


In [ ]:
# Figure 20: Pre vs Post rasters
fig, axes = plt.subplots(2,1,figsize=(13,8),sharex=True)
for row,(spk_arr,ttl) in enumerate([(spk0,'Pre-tuning (initial)'),(best_spk,'Post-tuning (best)')]):
    for ct in ['E','PV','SST','VIP']:
        for nid in type_indices[ct]:
            sm = spk_arr[:,nid]>0.5
            axes[row].scatter(T_AXIS[sm],[nid]*sm.sum(),c=COLORS[ct],s=1.0,alpha=0.7)
    axes[row].axvline(0,color='r',lw=0.8,ls='--')
    axes[row].axvline(500,color='orange',lw=0.8,ls='--',alpha=0.8)
    axes[row].set_title(f'Figure 20 -- {ttl} [computational scaffold]')
    axes[row].set_ylabel('Neuron ID')
axes[-1].set_xlabel('Time (ms)')
plt.tight_layout()
savefig('20_pre_vs_post_raster', fig)
print('Figure 20 saved.')


In [ ]:
# Figures 21-22: TFR and bandpower comparison
def get_tfr(src_arr, pos_jax, nc, dt):
    fld = project_laminar_sources(src_arr, pos_jax, n_contacts=nc, width=0.08)
    lm = np.array(fld.lfp_proxy).mean(axis=1)
    fsh = 1000.0/dt; nps = min(256, len(lm)//4)
    f_, t_, S_ = scipy_signal.spectrogram(lm, fs=fsh, nperseg=nps,
                                           noverlap=nps-int(nps*0.9), scaling='density')
    return f_, t_*1000+T_AXIS[0], S_

f_pre, t_pre, Sxx_pre = get_tfr(src0, positions_jax, n_contacts, DT_MS)
f_post, t_post, Sxx_post = get_tfr(best_src, positions_jax, n_contacts, DT_MS)
fm = f_pre <= 80

# Figure 21: Pre/Post TFR
fig, axes = plt.subplots(1,2,figsize=(14,5),sharey=True)
for ax, Sx, tm, lbl in [(axes[0],Sxx_pre,t_pre,'Pre-tuning'),(axes[1],Sxx_post,t_post,'Post-tuning')]:
    S_db = 10*np.log10(np.maximum(Sx[fm,:],1e-30))
    vm=np.percentile(S_db,99); vn=vm-25
    im=ax.pcolormesh(tm,f_pre[fm],S_db,cmap='viridis',vmin=vn,vmax=vm,shading='gouraud')
    plt.colorbar(im,ax=ax,label='dB')
    ax.axvline(0,color='w',lw=1.0,ls='--'); ax.axvline(500,color='yellow',lw=0.8,ls='--',alpha=0.7)
    ax.set_xlabel('Time (ms)'); ax.set_ylabel('Freq (Hz)')
    ax.set_title(f'Figure 21 -- {lbl}\n[LFP-proxy TFR | NOT calibrated]')
plt.tight_layout()
savefig('21_pre_vs_post_tfr', fig)

# Figure 22: Pre/Post bandpower
psd_pre = 10*np.log10(np.maximum(Sxx_pre[fm,:].mean(axis=1),1e-30))
psd_post = 10*np.log10(np.maximum(Sxx_post[fm,:].mean(axis=1),1e-30))
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(f_pre[fm], psd_pre, color='steelblue', lw=2, label='Pre-tuning')
ax.plot(f_pre[fm], psd_post, color='red', lw=2, ls='--', label='Post-tuning')
ax.axvspan(8,30,alpha=0.1,color='blue',label='Alpha/beta')
ax.axvspan(30,80,alpha=0.1,color='purple',label='Gamma')
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Power (dB)')
ax.set_title('Figure 22 -- Pre vs Post Tuning Bandpower\n'
             '[LFP-proxy PSD | NOT empirically calibrated | computational scaffold]')
ax.legend(fontsize=8)
savefig('22_pre_vs_post_bandpower', fig)
print('Figures 21-22 saved. All 22 figures complete.')


---
## Export: Metrics, Tuning History, Manifest


In [ ]:
# tutorial_metrics.csv
rows_m = []
for nid in range(N_TOTAL):
    ct = labels_ctx[nid]
    d = float(positions_ctx[nid, 2])
    layer = 'L6'
    for ln,(lo,hi) in LAYER_BOUNDS.items():
        if lo <= d < hi: layer=ln; break
    t_ev = (T_AXIS >= 0) & (T_AXIS < 500)
    fr_sp = float(spk_ctx_np[:,nid].sum())/(DURATION_MS/1000)
    fr_ev = float(best_spk[t_ev,nid].sum())/(500/1000)
    vc = v_ctx_np[:,nid]
    rows_m.append({'neuron_id':nid,'cell_type':ct,'layer':layer,'depth':round(d,4),
                   'firing_rate_spont_hz':round(fr_sp,3),'firing_rate_evoked_hz':round(fr_ev,3),
                   'v_min_mV':round(float(vc.min()),3),'v_max_mV':round(float(vc.max()),3),
                   'v_mean_mV':round(float(vc.mean()),3),'v_std_mV':round(float(vc.std()),3),
                   'source_calibration_status':'uncalibrated_izhikevich_native_current',
                   'claim_level':'computational_scaffold'})

metrics_df = pd.DataFrame(rows_m)
metrics_df.to_csv('tutorial_metrics.csv', index=False)
print(f'tutorial_metrics.csv: {len(metrics_df)} rows')
print(metrics_df.head(3).to_string())


In [ ]:
# tuning_history.csv
tuning_df.to_csv('tuning_history.csv', index=False)
print(f'tuning_history.csv: {len(tuning_df)} rows')
print(tuning_df.to_string())


In [ ]:
# tutorial_manifest.json
def sf(x):
    if x is None: return None
    if isinstance(x, float) and (x!=x or x==float('inf') or x==float('-inf')): return None
    return round(float(x), 6)

figure_list = [
    '01_single_neuron_voltage','02_single_neuron_spikes','03_single_neuron_source_proxy',
    '04_all_to_all','05_sparse_mask','06_pop_raster','07_pop_rate',
    '08_layout','09_spont_raster','10_spont_rate','11_lfp_proxy','12_csd_proxy',
    '13_tfr','14_bandpower','15_evoked_raster','16_evoked_lfp',
    '17_evoked_vs_spont_bandpower','18_loss_curve','19_param_trajectory',
    '20_pre_vs_post_raster','21_pre_vs_post_tfr','22_pre_vs_post_bandpower',
]

manifest_data = {
    'claim_level': 'computational_scaffold',
    'source_calibration_status': 'uncalibrated_izhikevich_native_current',
    'source_projection_mode': 'proxy_no_field_solve',
    'readout_status': 'LFP-proxy and CSD-proxy readouts',
    'n_total': N_TOTAL,
    'dt_ms': DT_MS,
    'duration_ms': DURATION_MS,
    'windows_ms': {'baseline':[-500,0],'event':[0,500],'post':[500,1000]},
    'cell_type_counts': {'E':N_E,'PV':N_PV,'SST':N_SST,'VIP':N_VIP},
    'connectivity': {'sparsity':0.1,'signed_ei':True},
    'readout_operators': ['spk','rate','source_proxy','lfp_proxy','csd_proxy','tfr','bandpower'],
    'objective_weights': OBJ_WEIGHTS,
    'n_opt_steps': N_OPT,
    'best_loss': sf(best_loss),
    'best_metrics': {
        'rates_hz': {k: sf(v) for k,v in best_m['rates'].items()},
        'ab_db': sf(best_m['ab_db']),
        'gm_db': sf(best_m['gm_db']),
        'sync': sf(best_m['sync']),
    },
    'best_parameters': {k: sf(v) for k,v in best_p.items()},
    'figure_list': figure_list,
    'version': 'jaxfne-colab-tutorial-v1',
    'truth_mode': 'truth_safe_unverified',
    'tutorial_purpose': 'tutorial_exploratory_not_biological_truth',
}

mj = json.dumps(manifest_data, indent=2)
json.loads(mj)  # validate no NaN
with open('tutorial_manifest.json','w') as f: f.write(mj)
print('tutorial_manifest.json saved.')
print(f'  claim_level: {manifest_data["claim_level"]}')
print(f'  best_loss:   {manifest_data["best_loss"]}')
print(f'  version:     {manifest_data["version"]}')


In [ ]:
# Final validation
import os, json
print('=== VALIDATION CHECKLIST ===')

figure_list_v = [
    '01_single_neuron_voltage','02_single_neuron_spikes','03_single_neuron_source_proxy',
    '04_all_to_all','05_sparse_mask','06_pop_raster','07_pop_rate',
    '08_layout','09_spont_raster','10_spont_rate','11_lfp_proxy','12_csd_proxy',
    '13_tfr','14_bandpower','15_evoked_raster','16_evoked_lfp',
    '17_evoked_vs_spont_bandpower','18_loss_curve','19_param_trajectory',
    '20_pre_vs_post_raster','21_pre_vs_post_tfr','22_pre_vs_post_bandpower',
]
missing = []
for name in figure_list_v:
    for ext in ['png','svg']:
        p = os.path.join(FIG_DIR, f'{name}.{ext}')
        if not os.path.exists(p) or os.path.getsize(p)==0:
            missing.append(p)

if missing:
    print(f'[FAIL] Missing figures: {missing}')
else:
    print(f'[PASS] All {len(figure_list_v)*2} figures present and nonzero')

for csv_n in ['tutorial_metrics.csv','tuning_history.csv']:
    try:
        df_ = pd.read_csv(csv_n)
        print(f'[PASS] {csv_n}: {len(df_)} rows')
    except Exception as e:
        print(f'[FAIL] {csv_n}: {e}')

try:
    with open('tutorial_manifest.json') as f: m_ = json.load(f)
    assert m_['claim_level'] == 'computational_scaffold'
    assert m_['source_calibration_status'] == 'uncalibrated_izhikevich_native_current'
    assert m_['version'] == 'jaxfne-colab-tutorial-v1'
    print('[PASS] tutorial_manifest.json: parsed, claim gates immutable')
except Exception as e:
    print(f'[FAIL] manifest: {e}')

print('[PASS] Claim discipline: proxy/LFP-proxy/CSD-proxy/computational_scaffold language used throughout')

print('\n=== BEST TUNING METRICS ===')
for ct in ['E','PV','SST','VIP']:
    print(f'  {ct} rate: {best_m["rates"][ct]:.2f} Hz  (target: {TARGET_RATES[ct]} Hz)')
print(f'  Alpha/beta: {best_m["ab_db"]:.2f} dB')
print(f'  Gamma:      {best_m["gm_db"]:.2f} dB')
print(f'  Sync proxy: {best_m["sync"]:.3f}')
print(f'  Best loss:  {best_loss:.4f}')

print('\n=== KNOWN LIMITATIONS ===')
print('  - CPU-only, n=100 neurons (toy scale)')
print('  - No real field solver (LFP-proxy/CSD-proxy readouts only)')
print('  - 15-step demo optimization, NOT converged')
print('  - Izhikevich native current is toy-scale, NOT calibrated to physical units')


---
## Summary and Claim Gates

You have completed the **jaxfne Computational Biophysics Tutorial**.

### What You Built

| Part | Key artifacts |
|------|---------------|
| Part 1 | Single E neuron voltage + spikes + source proxy (figs 01-03) |
| Part 2 | N=50 population raster + firing rate + connectivity (figs 04-07) |
| Part 3 | Laminar column, spontaneous/evoked, LFP-proxy, CSD-proxy, TFR, bandpower (figs 08-17) |
| Part 4 | Loss curve, parameter trajectory, pre/post comparison (figs 18-22) |
| Export | tutorial_metrics.csv, tuning_history.csv, tutorial_manifest.json |

### Claim Gates (Immutable)

| Field | Value |
|-------|-------|
| claim_level | computational_scaffold |
| source_calibration_status | toy_scale_not_empirical |
| source_projection_mode | proxy_no_field_solve |
| readout_status | LFP-proxy and CSD-proxy readouts |
| truth_mode | truth_safe_unverified |


*jaxfne-colab-tutorial-v1 | truth_safe_unverified | tutorial_exploratory_not_biological_truth*
